In [ ]:
import pandas as pd

# Load dataset
file_path = "canis_proteins.csv"
df = pd.read_csv(file_path)

# Display basic information
print(df.info())
print(df.head())  # Show the first few rows


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Define k-mer size
k = 3  

# Function to split sequences into k-mers
def get_kmers(sequence, k=3):
    return [" ".join([sequence[i:i+k] for i in range(len(sequence)-k+1)])]

# Apply to all sequences
sequences_kmers = df['Sequence'].apply(lambda x: get_kmers(x, k))

# Convert to a format suitable for CountVectorizer
sequences_kmers = [" ".join(kmer) for kmer in sequences_kmers]

# Use CountVectorizer for efficient k-mer counting
vectorizer = CountVectorizer(analyzer=lambda x: x.split(), max_features=1000)  # Limit to 1000 k-mers
kmer_features = vectorizer.fit_transform(sequences_kmers)

# Convert to DataFrame
kmer_df = pd.DataFrame(kmer_features.toarray(), columns=vectorizer.get_feature_names_out())

# Display first few rows
print(kmer_df.head())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compute sequence lengths
sequence_lengths = df['Sequence'].apply(len)

# Plot the distribution
sns.histplot(sequence_lengths, bins=30, kde=True)
plt.xlabel("Sequence Length")
plt.ylabel("Frequency")
plt.title("Distribution of Protein Sequence Lengths")
plt.show()


In [ ]:
from sklearn.decomposition import PCA

# Apply PCA to reduce to 2 components
pca = PCA(n_components=2)
kmer_pca = pca.fit_transform(kmer_df)

# Display the transformed shape
print("PCA transformed shape:", kmer_pca.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Scale the k-mer feature data
scaler = StandardScaler()
scaled_kmer = scaler.fit_transform(kmer_df)

# Apply PCA with 10 components
pca = PCA(n_components=10)
kmer_pca = pca.fit_transform(scaled_kmer)

# Check new shape
print("New PCA transformed shape:", kmer_pca.shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Try different K values
inertia = []
K_range = range(2, 10)  # Checking clusters from 2 to 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(kmer_pca)
    inertia.append(kmeans.inertia_)  # Sum of squared distances to cluster centers

# Plot the elbow curve
plt.plot(K_range, inertia, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Distortion Score)")
plt.title("Elbow Method for Optimal K")
plt.show()


In [ ]:
optimal_k = 4  # Replace with the best K from the elbow method

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(kmer_pca)

print(df[['Protein', 'Cluster']].head())  # Check if different clusters are assigned


In [ ]:
df.to_csv("canis_proteins.csv", index=False)
print("Processed dataset saved successfully!")


In [ ]:
df['Cluster'].value_counts().plot(kind='bar', color=['blue', 'orange', 'green', 'red'])
plt.xlabel("Cluster")
plt.ylabel("Number of Sequences")
plt.title("Protein Sequences per Cluster")
plt.show()


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
tsne_pca = tsne.fit_transform(kmer_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=tsne_pca[:,0], y=tsne_pca[:,1], hue=df['Cluster'], palette='viridis')
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.title("t-SNE Visualization of Protein Clusters")
plt.legend()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN



In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Assuming `df_kmer` is your k-mer frequency dataframe (excluding non-numeric columns)
df_kmer = df.drop(["Protein", "Sequence"], axis=1, errors="ignore")

# Scale the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_kmer)

# Apply DBSCAN clustering
dbscan = DBSCAN(eps=1.2, min_samples=3)
df['DBSCAN_Cluster'] = dbscan.fit_predict(scaled_data)

# Print cluster counts
print(df['DBSCAN_Cluster'].value_counts())


In [ ]:
dbscan = DBSCAN(eps=0.7, min_samples=3)
df['DBSCAN_Cluster'] = dbscan.fit_predict(scaled_data)

print(df['DBSCAN_Cluster'].value_counts())


In [ ]:
print(scaled_data.shape)


In [ ]:
# Check available numeric columns
print(df.dtypes)

# Select numeric columns (excluding cluster labels)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['Cluster', 'DBSCAN_Cluster']]  # Remove clustering labels

# Ensure at least 2 features exist
if len(numeric_cols) < 2:
    print("Not enough numeric features for PCA!")
else:
    # Scale selected numeric columns
    scaled_data = StandardScaler().fit_transform(df[numeric_cols])
    print(f"Scaled data shape: {scaled_data.shape}")

    # Apply PCA
    pca = PCA(n_components=2)
    pca_data = pca.fit_transform(scaled_data)
    print(f"PCA data shape: {pca_data.shape}")  # Should be (35884, 2)


In [ ]:
# Extract sequence length as a numeric feature
df['Sequence_Length'] = df['Sequence'].apply(len)

# Convert amino acid composition into features (example: count of each amino acid)
import collections
amino_acids = "ACDEFGHIKLMNPQRSTVWY"

for aa in amino_acids:
    df[f'Freq_{aa}'] = df['Sequence'].apply(lambda seq: seq.count(aa) / len(seq) if len(seq) > 0 else 0)

# Select new numeric columns
numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and col not in ['Cluster', 'DBSCAN_Cluster']]

# Scale the data
scaled_data = StandardScaler().fit_transform(df[numeric_cols])

# Apply PCA
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

print(f"PCA data shape: {pca_data.shape}")  # Should be (35884, 2)


In [ ]:
import matplotlib.pyplot as plt

# Create DataFrame for PCA results
df_pca = pd.DataFrame(pca_data, columns=['PCA1', 'PCA2'])
df_pca['DBSCAN_Cluster'] = df['DBSCAN_Cluster']  # Add cluster labels

# Plot
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df_pca['PCA1'], df_pca['PCA2'], c=df_pca['DBSCAN_Cluster'], cmap='viridis', alpha=0.5)
plt.colorbar(label='Cluster')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('DBSCAN Clusters Visualized with PCA')
plt.show()


In [ ]:
# Count the number of points in each DBSCAN cluster
cluster_counts = df['DBSCAN_Cluster'].value_counts().sort_index()

print("Cluster Distribution:")
print(cluster_counts)

# Check if there is a large noise cluster (-1)
if -1 in cluster_counts:
    print(f"\nNoise Cluster (-1) contains {cluster_counts[-1]} points.")
else:
    print("\nNo noise cluster detected.")


In [ ]:
import plotly.express as px
import pandas as pd

# Create DataFrame for PCA visualization
df_pca = pd.DataFrame(pca_data, columns=['PCA1', 'PCA2'])
df_pca['Cluster'] = df['DBSCAN_Cluster']
df_pca['Protein'] = df['Protein']  # Assuming Protein column exists

# Interactive scatter plot
fig = px.scatter(
    df_pca, x='PCA1', y='PCA2', color='Cluster', 
    hover_data=['Protein', 'Cluster'], title='Interactive PCA Clustering'
)
fig.show()


In [ ]:
pip install scikit-learn


In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

# Compute k-distance for k=min_samples (e.g., 4)
k = 4
nbrs = NearestNeighbors(n_neighbors=k).fit(scaled_data)
distances, _ = nbrs.kneighbors(scaled_data)

# Sort and plot the k-distances
sorted_distances = np.sort(distances[:, k-1])
plt.plot(sorted_distances)
plt.xlabel("Points sorted by distance")
plt.ylabel(f"{k}-NN Distance")
plt.title("K-distance plot for DBSCAN eps selection")
plt.show()


In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
df['DBSCAN_Cluster'] = dbscan.fit_predict(scaled_data)


In [ ]:
from sklearn.metrics import silhouette_score

# Check if there are multiple clusters (Silhouette Score needs at least 2 clusters)
unique_clusters = df['DBSCAN_Cluster'].nunique()

if unique_clusters > 1:
    # Exclude noise (-1) if present
    valid_clusters = df[df['DBSCAN_Cluster'] != -1]
    
    # Compute Silhouette Score
    score = silhouette_score(scaled_data[valid_clusters.index], valid_clusters['DBSCAN_Cluster'])
    print(f"Silhouette Score: {score:.4f}")
else:
    print("Silhouette Score cannot be computed with only one cluster.")


In [ ]:
pip install streamlit


In [ ]:
pip show streamlit


In [ ]:
pip install biopython


In [ ]:
df[df['Cluster'] == 0][['Protein', 'Sequence']].to_csv("cluster_0.fasta", index=False, header=False, sep="\n")


In [ ]:
import pandas as pd

# Ensure df exists before running this
df[['Protein', 'Sequence']].to_csv("C:/Users/sanja/custom_db.fasta", index=False, header=False, sep="\n")

print("custom_db.fasta created successfully!")



In [ ]:
with open("C:/Users/sanja/custom_db.fasta", "r") as file:
    for i in range(5):  # Print first 5 lines
        print(file.readline().strip())


In [ ]:
with open("C:/Users/sanja/custom_db.fasta", "w") as file:
    for index, row in df.iterrows():
        file.write(f">{row['Protein']}\n{row['Sequence']}\n")  # Ensure '>' is added before Protein names

print("✅ custom_db.fasta has been reformatted correctly!")


In [ ]:
with open("C:/Users/sanja/custom_db.fasta", "r") as file:
    for i in range(5):  # Print first 5 lines
        print(file.readline().strip())



In [ ]:
with open(r"C:\Users\sanja\cluster_0.fasta", "w") as file:  # Use raw string format
    for index, row in df[df['Cluster'] == 0].iterrows():
        file.write(f">{row['Protein']}\n{row['Sequence']}\n")  # Ensure '>' is added

print("✅ cluster_0.fasta has been created successfully!")


In [ ]:
import pandas as pd

# Load BLAST results into a DataFrame
blast_results = pd.read_csv("C:/Users/sanja/results_cluster_0.txt", sep="\t", header=None)

# Assign column names (BLAST output format 6)
blast_results.columns = [
    "Query", "Subject", "Identity", "Alignment_Length", "Mismatches",
    "Gaps", "Query_Start", "Query_End", "Subject_Start", "Subject_End",
    "E-Value", "Bit Score"
]

# Display top results
print(blast_results.head())


In [ ]:
filtered_results = blast_results[(blast_results["Identity"] > 50) & (blast_results["E-Value"] < 1e-5)]
print(filtered_results)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.histplot(blast_results["Identity"], bins=30, kde=True)
plt.xlabel("Sequence Identity (%)")
plt.ylabel("Frequency")
plt.title("BLAST Identity Distribution")
plt.show()


In [ ]:
best_hits = blast_results.sort_values("Bit Score", ascending=False).groupby("Query").first()
print(best_hits)


In [ ]:
import collections

amino_acids = "ACDEFGHIKLMNPQRSTVWY"

# Compute amino acid frequencies per cluster
cluster_aa_freq = {}

for cluster in df['Cluster'].unique():
    cluster_data = df[df['Cluster'] == cluster]['Sequence']
    aa_counts = collections.Counter("".join(cluster_data))
    total_aa = sum(aa_counts.values())
    cluster_aa_freq[cluster] = {aa: aa_counts[aa] / total_aa for aa in amino_acids}

# Convert to DataFrame
import pandas as pd
aa_df = pd.DataFrame(cluster_aa_freq).T

# Plot heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(aa_df, cmap="coolwarm", annot=True, fmt=".2f")
plt.xlabel("Amino Acids")
plt.ylabel("Clusters")
plt.title("Amino Acid Composition per Cluster")
plt.show()


In [ ]:
print(df_pca.columns)


In [ ]:
noise_points = df_pca[df_pca['Cluster'] == -1]


In [ ]:
print(df_pca.columns)


In [ ]:
from sklearn.metrics import silhouette_score

# Check if DBSCAN found more than 1 cluster
unique_clusters = df['DBSCAN_Cluster'].nunique()

if unique_clusters > 1:
    # Exclude noise (-1)
    valid_clusters = df[df['DBSCAN_Cluster'] != -1]

    # Ensure there are at least 2 valid clusters after removing noise
    if valid_clusters['DBSCAN_Cluster'].nunique() > 1:
        # Compute Silhouette Score
        score = silhouette_score(scaled_data[valid_clusters.index], valid_clusters['DBSCAN_Cluster'].astype(int))
        print(f"Silhouette Score: {score:.4f}")
    else:
        print("Silhouette Score cannot be computed: Only one valid cluster found after removing noise.")
else:
    print("Silhouette Score cannot be computed: DBSCAN detected only one cluster.")

In [ ]:
df_pca['Cluster'] = dbscan.fit_predict(df_pca[['PCA1', 'PCA2']])


In [ ]:
import matplotlib.pyplot as plt

# Plot noise points (-1 cluster)
noise_points = df_pca[df_pca['DBSCAN_Cluster'] == -1]
plt.scatter(noise_points['PCA1'], noise_points['PCA2'], color='red', label='Noise', alpha=0.6)

# Plot each cluster with a unique color
for cluster in df_pca['DBSCAN_Cluster'].unique():
    if cluster != -1:  # Skip noise points
        cluster_points = df_pca[df_pca['DBSCAN_Cluster'] == cluster]
        plt.scatter(cluster_points['PCA1'], cluster_points['PCA2'], label=f'Cluster {cluster}', alpha=0.6)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.title('DBSCAN Clustering Results (PCA1 vs PCA2)')
plt.show()


In [ ]:
print(df['Cluster'].value_counts())  # Check the count of noise points (Cluster = -1)


In [ ]:
noise_points = df_pca[df_pca['DBSCAN_Cluster'] == -1]

plt.scatter(noise_points['PCA1'], noise_points['PCA2'], color='red', label='Noise', alpha=0.6)
plt.legend()
plt.show()


In [ ]:
print(df_pca.columns)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

for cluster in df_pca['DBSCAN_Cluster'].unique():
    cluster_points = df_pca[df_pca['DBSCAN_Cluster'] == cluster]
    plt.scatter(cluster_points['PCA1'], cluster_points['PCA2'], label=f'Cluster {cluster}', alpha=0.6)

plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.legend()
plt.title('DBSCAN Clustering on PCA Data')
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)  # Adjust these values
clusters = dbscan.fit_predict(pca_data)

df_pca['DBSCAN_Cluster'] = clusters


In [ ]:
noise_points = df_pca[df_pca['DBSCAN_Cluster'] == -1]
print(f"Number of noise points: {len(noise_points)}")


In [ ]:
dbscan = DBSCAN(eps=0.6, min_samples=4)  # Try increasing eps or reducing min_samples


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df_pca['PCA1'], df_pca['PCA2'], c=df_pca['DBSCAN_Cluster'], cmap='viridis', alpha=0.5)
plt.scatter(noise_points['PCA1'], noise_points['PCA2'], color='red', label='Noise')
plt.legend()
plt.title("DBSCAN Clustering with Noise Points")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))

# Plot each cluster
for cluster in df_pca['DBSCAN_Cluster'].unique():
    cluster_points = df_pca[df_pca['DBSCAN_Cluster'] == cluster]
    ax.scatter(cluster_points['PCA1'], cluster_points['PCA2'], label=f'Cluster {cluster}', alpha=0.6)

ax.set_xlabel('PCA1')
ax.set_ylabel('PCA2')
ax.set_title('DBSCAN Clustering Results (2D)')

plt.legend(bbox_to_anchor=(1.1, 1))
plt.show()


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)  # Extract 3 principal components
df_pca_3d = pca.fit_transform(scaled_data)  # Apply PCA on your scaled data

# Convert to DataFrame
df_pca_3d = pd.DataFrame(df_pca_3d, columns=['PCA1', 'PCA2', 'PCA3'])
df_pca_3d['DBSCAN_Cluster'] = clusters  # Add clustering labels

# Now you can plot in 3D!
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

for cluster in df_pca_3d['DBSCAN_Cluster'].unique():
    cluster_points = df_pca_3d[df_pca_3d['DBSCAN_Cluster'] == cluster]
    ax.scatter(cluster_points['PCA1'], cluster_points['PCA2'], cluster_points['PCA3'], label=f'Cluster {cluster}', alpha=0.6)

ax.set_xlabel('PCA1')
ax.set_ylabel('PCA2')
ax.set_zlabel('PCA3')
ax.set_title('DBSCAN Clustering Results (3D)')

plt.legend(bbox_to_anchor=(1.1, 1))
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

z = np.linspace(0, 10, 1000)
x = np.sin(z * 2 * np.pi)
y = np.cos(z * 2 * np.pi)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot(x, y, z)
plt.title("Simple DNA-like Spiral")
plt.show()

In [ ]:
# Load your dataset if not already loaded
import pandas as pd
df = pd.read_csv("canis_proteins.csv")  # or use the loaded df in memory

# Pick the first sequence (or any index)
sequence = df['Sequence'].iloc[0]

# Optional: Filter to keep only DNA characters (ATGC)
sequence = ''.join([s for s in sequence if s in "ATGC"])

In [ ]:
import numpy as np

# Number of bases in the sequence
num_points = len(sequence)

# Helix shape parameters
theta = np.linspace(0, 4 * np.pi, num_points)  # 2 full turns
z = np.linspace(0, 10, num_points)             # height from bottom to top
x = np.sin(theta)                              # circle x
y = np.cos(theta)                              # circle y

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Color for each base
colors = {'A': 'red', 'T': 'blue', 'G': 'green', 'C': 'yellow'}

# Create 3D plot
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

# Plot the spiral backbone
ax.plot(x, y, z, color='gray', alpha=0.6)

# Plot each base as a colored dot
for i in range(num_points):
    base = sequence[i]
    ax.scatter(x[i], y[i], z[i], color=colors.get(base, 'black'), s=30)

ax.set_title("DNA Spiral Visualization from Sequence")
plt.tight_layout()
plt.show()

In [ ]:
# Simplified codon table: one codon per amino acid
amino_acid_to_codon = {
    'A': 'GCT', 'R': 'CGT', 'N': 'AAT', 'D': 'GAT', 'C': 'TGT',
    'Q': 'CAA', 'E': 'GAA', 'G': 'GGT', 'H': 'CAT', 'I': 'ATT',
    'L': 'TTA', 'K': 'AAA', 'M': 'ATG', 'F': 'TTT', 'P': 'CCT',
    'S': 'TCT', 'T': 'ACT', 'W': 'TGG', 'Y': 'TAT', 'V': 'GTT'
}

In [ ]:
# Pick a protein sequence from your dataframe
protein_seq = df['Sequence'].iloc[0]

# Convert to DNA using the codon table
dna_seq = ''.join([amino_acid_to_codon.get(aa, '') for aa in protein_seq])
print("DNA sequence:", dna_seq)

In [ ]:
from matplotlib import animation
from IPython.display import HTML

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Convert the DNA string into spiral coordinates
num_points = len(dna_seq)
theta = np.linspace(0, 4 * np.pi, num_points)
z = np.linspace(0, 10, num_points)
x = np.sin(theta)
y = np.cos(theta)

# Color map
colors = {'A': 'red', 'T': 'blue', 'G': 'green', 'C': 'yellow'}

# Setup figure
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Function to draw each frame
def update(frame):
    ax.clear()
    ax.set_title("Animated DNA Spiral")
    ax.plot(x, y, z, color='gray', alpha=0.3)
    for i in range(frame):
        base = dna_seq[i]
        ax.scatter(x[i], y[i], z[i], color=colors.get(base, 'black'), s=40)

# Create animation
ani = animation.FuncAnimation(fig, update, frames=min(num_points, 150), interval=60)

# Display animation
HTML(ani.to_jshtml())

In [ ]:
import pandas as pd
import numpy as np

# Simulated enrichment results
enrichment_df = pd.DataFrame({
    'GO Term': ['GO:0008150', 'GO:0003674', 'GO:0005575', 'GO:0044237', 'GO:0009987'],
    'Description': ['biological_process', 'molecular_function', 'cellular_component', 'cellular metabolic process', 'cellular process'],
    'p-value': [0.001, 0.005, 0.02, 0.04, 0.03],
    'Gene Count': [45, 30, 60, 25, 40]
})


In [ ]:
enrichment_df['-log10(p-value)'] = -np.log10(enrichment_df['p-value'])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=enrichment_df,
    x='Gene Count',
    y='Description',
    size='-log10(p-value)',
    hue='-log10(p-value)',
    sizes=(100, 1000),
    palette='viridis',
    legend='full'
)
plt.title("GO Term Enrichment Bubble Plot")
plt.xlabel("Gene Count")
plt.ylabel("GO Description")
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px

polar_data = pd.DataFrame({
    'Category': ['Metabolism', 'Signal Transduction', 'Transport', 'Structure', 'Defense'],
    'Value': [20, 35, 15, 25, 30]
})


In [ ]:
fig = px.bar_polar(
    polar_data,
    r='Value',
    theta='Category',
    color='Category',
    template='plotly_dark',
    title="Polar Bar Plot of Functional Categories"
)
fig.show()


In [ ]:
pip install circlify


In [ ]:
import circlify

# Sample GO enrichment results
go_terms = [
    {'id': 'GO:0006412', 'label': 'Translation', 'value': 30},
    {'id': 'GO:0008152', 'label': 'Metabolism', 'value': 25},
    {'id': 'GO:0009987', 'label': 'Cellular Process', 'value': 20},
    {'id': 'GO:0055114', 'label': 'Oxidation-Reduction', 'value': 15},
    {'id': 'GO:0005975', 'label': 'Carbohydrate Metabolism', 'value': 10}
]

# Create layout
circles = circlify.circlify(
    [item['value'] for item in go_terms],
    show_enclosure=False,
    target_enclosure=circlify.Circle(x=0, y=0, r=1)
)

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
for circle, term in zip(circles, go_terms):
    x, y, r = circle.x, circle.y, circle.r
    ax.add_patch(plt.Circle((x, y), r, alpha=0.5))
    ax.text(x, y, term['label'], ha='center', va='center', fontsize=9)
plt.title("GO Chord-Like Plot (Circle Packing)")
plt.show()


In [ ]:
import pandas as pd

# Create mock differential expression data
data = {
    'gene': ['GeneA', 'GeneB', 'GeneC', 'GeneD', 'GeneE', 'GeneF', 'GeneG'],
    'log2FC': [2.1, -1.5, 0.5, -2.3, 1.8, -0.7, 0.2],
    'pvalue': [0.0001, 0.01, 0.5, 0.0003, 0.001, 0.4, 0.7]
}

df = pd.DataFrame(data)
df['-log10(pvalue)'] = -np.log10(df['pvalue'])
df['significant'] = (df['pvalue'] < 0.05) & (abs(df['log2FC']) > 1)
df.head()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='log2FC', y='-log10(pvalue)', hue='significant',
                palette={True: 'red', False: 'grey'}, legend=False)

plt.axhline(y=-np.log10(0.05), color='blue', linestyle='--')
plt.axvline(x=1, color='blue', linestyle='--')
plt.axvline(x=-1, color='blue', linestyle='--')

plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Volcano Plot')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import circlify

# Sample data: GO terms and the number of associated proteins
go_terms = [
    {'id': 'GO:0003700', 'label': 'DNA-binding TF activity', 'value': 2},
    {'id': 'GO:0006355', 'label': 'Regulation of transcription', 'value': 2},
    {'id': 'GO:0005634', 'label': 'Nucleus', 'value': 1},
    # Add more GO terms as needed
]

# Create circle packing layout
circles = circlify.circlify(
    [item['value'] for item in go_terms],
    show_enclosure=False,
    target_enclosure=circlify.Circle(x=0, y=0, r=1)
)

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
for circle, term in zip(circles, go_terms):
    x, y, r = circle.x, circle.y, circle.r
    ax.add_patch(plt.Circle((x, y), r, alpha=0.5))
    ax.text(x, y, term['label'], ha='center', va='center', fontsize=9)
plt.title("GO Chord-Like Plot (Circle Packing)")
plt.show()
